In [23]:
import json
from ollama import Client
from dotenv import load_dotenv

# Force reload the freshly updated .env file if needed
load_dotenv(override=True)

# Initialize the Ollama client (defaults to http://localhost:11434)
client = Client()

# Q1

In [2]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [4]:
from pydantic import BaseModel, Field

# 1. Define the structural output schema using Python 3.11 standard lowercase list
class QuestionItem(BaseModel):
    question: str = Field(description="A clear question based on the document content.")

class Questions(BaseModel):
    questions: list[QuestionItem] = Field(description="A list of generated questions.")

# 2. Add placeholder instructions if they aren't defined yet
if 'data_gen_instructions' not in locals():
    data_gen_instructions = """
    You are an expert course assistant. Read the provided document and generate 
    a series of distinct questions that can be completely answered using only 
    the text content found inside the document.
    """.strip()

In [5]:
import json
from evaluation_utils import llm_structured

# 1. Define the 3 specific homework files to filter for
target_filenames = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md"
]

input_token_counts = []

# 2. Iterate through parsed documents and filter for the target files
for doc in documents:
    filename = doc.get("filename")
    
    if filename in target_filenames:
        doc_content = doc.get("content", "")
        
        # Package into the expected json payload format
        user_prompt = json.dumps({"document": doc_content})
        
        # 3. Call your local structured utility function
        parsed_output, usage = llm_structured(
            client=None,
            instructions=data_gen_instructions, # Your data generation instructions variable
            user_prompt=user_prompt,
            output_type=Questions,              # Your notebook Pydantic model
            model="llama3.2:3b"                 # Or whichever local model you are using
        )
        
        # Extract the prompt input tokens via the custom utility wrapper property
        input_tokens = usage.prompt_token_count
        input_token_counts.append(input_tokens)
        
        print(f"File: {filename} | Input Tokens: {input_tokens}")

# 4. Calculate the average
if input_token_counts:
    average_input_tokens = sum(input_token_counts) / len(input_token_counts)
    print(f"\n--- Homework Result ---")
    print(f"Average number of input tokens: {average_input_tokens:.2f}")
else:
    print("No matching files found. Check your documents list names.")

File: 01-agentic-rag/lessons/01-intro.md | Input Tokens: 989
File: 01-agentic-rag/lessons/02-environment.md | Input Tokens: 1262
File: 01-agentic-rag/lessons/03-rag.md | Input Tokens: 1735

--- Homework Result ---
Average number of input tokens: 1328.67


# Q2

In [6]:
import pandas as pd
from ingest import load_faq_data, build_index

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [7]:
target_id = "74eb249bbf"

for doc in documents:
    if doc.get("id") == target_id:
        print("Available fields:", list(doc.keys()))
        # Print whichever fields might hold the path information
        for key in ["document", "path", "file", "filename"]:
            if key in doc:
                print(f"{key}: {doc[key]}")
        break

Available fields: ['id', 'course', 'section', 'question', 'answer']


In [8]:
# 1. Get the first question and search for it
q = ground_truth[0]["question"]
search_results = index.search(
    q,
    num_results=1,
    boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
    filter_dict={"course": "llm-zoomcamp"}
)

if search_results:
    top_hit = search_results[0]
    matched_question = top_hit["question"]
    matched_answer = top_hit["answer"]
    
    print(f"Matched Text: {matched_question}\n")

    # 2. Re-read or reference your gitsource reader to find which file holds this text
    # We use a unique variable name 'raw_files' so it doesn't conflict with your index documents
    raw_files = [file.parse() for file in reader.read()]
    
    # 3. Scan the markdown content for the matched FAQ text
    found_filename = None
    for file_obj in raw_files:
        content = file_obj.get("content", "")
        if matched_question in content or matched_answer in content:
            found_filename = file_obj.get("filename")
            break
            
    print(f"--- Homework Result ---")
    print(f"Filename of the first result: {found_filename}")

else:
    print("No search results found.")

Matched Text: I just discovered the course. Can I still join?

--- Homework Result ---
Filename of the first result: 01-agentic-rag/lessons/03-rag.md


# Q3

In [9]:
import inspect
from ingest import build_index

# Print the available methods on your index object
print("Available methods:", [method for method in dir(index) if not method.startswith('_')])

Available methods: ['date_df', 'date_fields', 'docs', 'fit', 'keyword_df', 'keyword_fields', 'load', 'numeric_df', 'numeric_fields', 'save', 'search', 'text_fields', 'text_matrices', 'vectorizers']


In [10]:
# 1. Import the library and initialize the model
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Define the text query you want to turn into an embedding vector
# (Feel free to change this text string to match your homework prompt)
query = "Can I still join the course after the start date?"

# 3. Generate the text embedding vector
v_query = model.encode(query)

# 4. Print results to verify it worked
print(f"Vector data type: {type(v_query)}")
print(f"Vector dimensions (length): {len(v_query)}")
print("\nFirst 5 values of your embedding vector:")
print(v_query[:5])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector data type: <class 'numpy.ndarray'>
Vector dimensions (length): 384

First 5 values of your embedding vector:
[ 0.02139041 -0.07397998  0.00142068  0.02138167  0.02451131]


In [17]:
from minsearch import VectorSearch  # <-- Changed from Index to VectorSearch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# 1. Initialize the embedding model
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Extract texts to embed safely
print("Preparing text data...")
texts = []
for doc in documents:
    question = doc.get("question", "")
    body_text = doc.get("text") or doc.get("answer") or ""
    texts.append(f"{question} {body_text}".strip())

# 3. Generate the embeddings in batches
print("Generating embedding vectors...")
vectors = []
batch_size = 32
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i : i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

# 4. Build and fit the VectorSearch index
print("Fitting vector index...")
# Dynamically pull available keyword fields present in your first document
keyword_fields = [k for k in ["course", "id"] if k in documents[0]]

# Initialize VectorSearch instead of Index
vector_index = VectorSearch(keyword_fields=keyword_fields)

# Fit expects: vectors first, then documents
vector_index.fit(vectors, documents)

print("🎉 vector_index successfully built, fitted, and ready to use!")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Preparing text data...
Generating embedding vectors...


100%|██████████| 4/4 [00:02<00:00,  1.67it/s]

Fitting vector index...
🎉 vector_index successfully built, fitted, and ready to use!


In [18]:
from minsearch import VectorSearch
import numpy as np

# 1. Grab the first question
q = ground_truth[0]["question"]

# 2. Generate the dense embedding vector for the question text
# (Ensure your embedding model/embedder from Module 2 is initialized)
# If your model variable is named 'embedder' or 'model', adapt the line below:
try:
    if 'embedder' in locals() or 'embedder' in globals():
        q_vector = embedder.encode(q)
    elif 'model' in locals() or 'model' in globals():
        q_vector = model.encode(q)
    else:
        # Fallback if you haven't loaded the model in this cell session yet
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("all-MiniLM-L6-v2")
        q_vector = model.encode(q)
except Exception as e:
    print(f"Error encoding query text into vector: {e}")
    q_vector = None

# 3. Perform the vector search using the VectorSearch index instance
# (Note: make sure you use your vector index variable name, e.g., vector_index)
vector_results = []
if q_vector is not None:
    try:
        # Ensure you pass the query vector, NOT the text string 'q'
        vector_results = vector_index.search(
            q_vector, 
            num_results=1, 
            filter_dict={"course": "llm-zoomcamp"}
        )
    except NameError:
        print("Please make sure your 'vector_index' object from your notebook is loaded and fitted.")

# 4. Map the top match back to its source filename
if vector_results:
    top_vector_hit = vector_results[0]
    matched_q = top_vector_hit.get("question")
    matched_a = top_vector_hit.get("answer")
    
    print(f"Vector Search Top Match Text:\n{matched_q}\n")

    # Reload git files to map the text to a filename
    raw_files = [file.parse() for file in reader.read()]
    
    found_filename = None
    for file_obj in raw_files:
        content = file_obj.get("content", "")
        if (matched_q and matched_q in content) or (matched_a and matched_a in content):
            found_filename = file_obj.get("filename")
            break
            
    print(f"--- Homework Result ---")
    print(f"Filename of the first vector result: {found_filename}")
else:
    print("Could not complete vector retrieval pipeline.")

Vector Search Top Match Text:
I just discovered the course. Can I still join?

--- Homework Result ---
Filename of the first vector result: 01-agentic-rag/lessons/03-rag.md


# Q4

In [19]:
from tqdm.auto import tqdm

# 1. Define the search function with the correct fields and boots
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

# 2. Define the helper to compute document relevance 
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

# 3. Define the hit rate evaluation function
def hit_rate(relevance_total):
    cnt = 0
    for line in relevance_total:
        if 1 in line:
            cnt = cnt + 1
    return cnt / len(relevance_total)

# 4. Loop through the entire ground truth dataset to compute total relevance
relevance_total = []
for q in tqdm(ground_truth):
    relevance = compute_relevance(q, text_search)
    relevance_total.append(relevance)

# 5. Calculate and print the final evaluation score
final_hit_rate = hit_rate(relevance_total)
print(f"\nCalculated Hit Rate: {final_hit_rate:.4f}")

  0%|          | 0/502 [00:00<?, ?it/s]


Calculated Hit Rate: 0.7410


# Q5

In [21]:
from tqdm.auto import tqdm


# 1. Update the vector search function (Removing filter_dict to match baseline evaluation)
def vector_search(query):
    # Encode the user query using the embedding model
    query_vector = model.encode(query)

    # Search across the entire index without constraining to a course filter
    return vector_index.search(query_vector, num_results=5)


# 2. Define the MRR evaluation function
def mrr(relevance_total):
    total_score = 0.0

    for line in relevance_total:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break  # Stop checking ranks once the correct document is found

    return total_score / len(relevance_total)


# 3. Compute relevance for the vector search across all ground truth data
relevance_total_vector = []
for q in tqdm(ground_truth):
    doc_id = q["document"]
    results = vector_search(q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    relevance_total_vector.append(relevance)

# 4. Calculate and print the final vector MRR
final_vector_mrr = mrr(relevance_total_vector)
print(f"\nCalculated Vector Search MRR: {final_vector_mrr:.4f}")

  0%|          | 0/502 [00:00<?, ?it/s]


Calculated Vector Search MRR: 0.7899


# Q6


In [24]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [25]:
from tqdm.auto import tqdm


# 1. Define the RRF hybrid search function that dynamically accepts k
def hybrid_search(query, k):
    # Get text search results
    boost_dict = {"question": 3.0, "section": 0.5}
    text_results = index.search(query, num_results=5, boost_dict=boost_dict)

    # Get vector search results
    query_vector = model.encode(query)
    vector_results = vector_index.search(query_vector, num_results=5)

    # Calculate RRF scores
    rrf_scores = {}

    for rank, doc in enumerate(text_results):
        doc_id = doc["id"]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)

    for rank, doc in enumerate(vector_results):
        doc_id = doc["id"]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)

    # Sort documents based on RRF scores
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

    # Reconstruct the top 5 document payloads matching minsearch formatting
    # mapping doc_idx back to retrieve original content dictionaries
    top_results = []
    for doc_id, _ in sorted_docs[:5]:
        top_results.append(doc_idx[doc_id])

    return top_results


# 2. Define standard MRR evaluation logic
def evaluate_hybrid_mrr(ground_truth, k):
    total_score = 0.0

    for q in ground_truth:
        doc_id = q["document"]
        results = hybrid_search(q["question"], k=k)

        for rank, d in enumerate(results):
            if d["id"] == doc_id:
                total_score += 1 / (rank + 1)
                break  # Stop checking ranks once the correct document is found

    return total_score / len(ground_truth)


# 3. Benchmark across requested k values
k_values = [1, 50, 100, 200]
results = {}

for k in k_values:
    print(f"Evaluating Hybrid Search with k={k}...")
    mrr_score = evaluate_hybrid_mrr(ground_truth, k)
    results[k] = mrr_score
    print(f"k={k} -> MRR: {mrr_score:.4f}\n")

# Find the best k value
best_k = min(results, key=lambda k: (-results[k], k))
print(f"🏆 Best k value (with tie-breaker applied): {best_k}")

Evaluating Hybrid Search with k=1...
k=1 -> MRR: 0.7150

Evaluating Hybrid Search with k=50...
k=50 -> MRR: 0.7090

Evaluating Hybrid Search with k=100...
k=100 -> MRR: 0.7090

Evaluating Hybrid Search with k=200...
k=200 -> MRR: 0.7090

🏆 Best k value (with tie-breaker applied): 1
